<a href="https://colab.research.google.com/github/1kaiser/Cerebras_wafer-_scale/blob/main/Cerebras%F0%9F%92%BFwafer_scale%F0%9F%93%80Reader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get update && apt-get install -y poppler-utils
!pip install uv
!uv pip install -q jupytext papermill pdf2image pypdf cerebras-cloud-sdk matplotlib numpy jinja2

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:9 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,307 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,258 kB]


In [4]:
!mkdir -p /content/chapters
!unzip lebo1dd.zip -d /content/chapters
!unzip kebo1dd.zip -d /content/chapters

Archive:  lebo1dd.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of lebo1dd.zip or
        lebo1dd.zip.zip, and cannot find lebo1dd.zip.ZIP, period.
Archive:  kebo1dd.zip
  inflating: /content/chapters/kebo1ps.pdf  
  inflating: /content/chapters/kebo101.pdf  
  inflating: /content/chapters/kebo102.pdf  
  inflating: /content/chapters/kebo103.pdf  
  inflating: /content/chapters/kebo104.pdf  
  inflating: /content/chapters/kebo105.pdf  
  inflating: /content/chapters/kebo106.pdf  
  inflating: /content/chapters/kebo107.pdf  
  inflating: /content/chapters/kebo108.pdf  
  inflating: /content/chapters/kebo109.pdf  
  inflating: /content/chapters/kebo110.pdf  
  inflating: /content/chapters/kebo111.pdf  
  inflating: /content/chapters/k

In [3]:
!mkdir -p /content/chemistry_chapters
!unzip kech2dd.zip -d /content/chemistry_chapters
!unzip kech1dd.zip -d /content/chemistry_chapters
!unzip lech2dd.zip -d /content/chemistry_chapters
!unzip lech1dd.zip -d /content/chemistry_chapters


Archive:  kech2dd.zip
  inflating: /content/chemistry_chapters/kech2an.pdf  
  inflating: /content/chemistry_chapters/kech2ps.pdf  
  inflating: /content/chemistry_chapters/kech201.pdf  
  inflating: /content/chemistry_chapters/kech202.pdf  
  inflating: /content/chemistry_chapters/kech203.pdf  
Archive:  kech1dd.zip
  inflating: /content/chemistry_chapters/kech1a1.pdf  
  inflating: /content/chemistry_chapters/kech1an.pdf  
  inflating: /content/chemistry_chapters/kech1ps.pdf  
  inflating: /content/chemistry_chapters/kech101.pdf  
  inflating: /content/chemistry_chapters/kech102.pdf  
  inflating: /content/chemistry_chapters/kech103.pdf  
  inflating: /content/chemistry_chapters/kech104.pdf  
  inflating: /content/chemistry_chapters/kech105.pdf  
  inflating: /content/chemistry_chapters/kech106.pdf  
Archive:  lech2dd.zip
  inflating: /content/chemistry_chapters/lech2an.pdf  
  inflating: /content/chemistry_chapters/lech2ps.pdf  
  inflating: /content/chemistry_chapters/lech201.pdf  

In [5]:
# @title main file
%%writefile active_rag_engine.py
import os
import pypdf
from pdf2image import convert_from_path
import base64
import io

class NCERTBookManager:
    """
    Manages a contiguously mapped, folder-driven textbook of PDFs.
    Ingests any arbitrary directory of PDF files, sorted alphabetically.
    Exposes a dynamic page-to-image converter with automatic offset translation.
    """
    def __init__(self, extract_dir="/content"):
        self.extract_dir = extract_dir
        self.available_files = []
        self._prepare_book()

    def _prepare_book(self):
        # 1. Verify and scan target directory
        if not os.path.exists(self.extract_dir):
            raise FileNotFoundError(f"❌ Target directory '{self.extract_dir}' does not exist.")

        all_files = os.listdir(self.extract_dir)
        pdf_files = sorted([f for f in all_files if f.endswith(".pdf")])

        if not pdf_files:
            raise FileNotFoundError(f"❌ No PDF files found inside target directory: {self.extract_dir}")

        self.available_files = pdf_files
        print(f"📂 Successfully mapped {len(self.available_files)} active textbook PDFs in {self.extract_dir}:")
        for f in self.available_files:
            print(f"  • {f}")

    def get_pdf_page_image_b64(self, pdf_name: str, page_num: int) -> str:
        """
        Renders page_num of a specific PDF file to a base64 PNG.
        Expects 1-based local page numbers, with safe boundary guards.
        """
        pdf_path = os.path.join(self.extract_dir, pdf_name)
        if not os.path.exists(pdf_path):
            # Try root folder fallback
            pdf_path = os.path.join("/content", pdf_name)
            if not os.path.exists(pdf_path):
                pdf_path = pdf_name
                if not os.path.exists(pdf_path):
                    raise FileNotFoundError(f"Could not locate PDF file: {pdf_name}")

        # Load local page size to check range and prevent crashes
        reader = pypdf.PdfReader(pdf_path)
        total_local_pages = len(reader.pages)

        local_page_num = page_num
        if local_page_num < 1:
            local_page_num = 1
        if local_page_num > total_local_pages:
            print(f"⚠️ Warning: Requested page {page_num} exceeds max pages ({total_local_pages}) for {pdf_name}. Defaulting to last page.")
            local_page_num = total_local_pages

        # Render PDF page (1-based index)
        images = convert_from_path(
            pdf_path,
            first_page=local_page_num,
            last_page=local_page_num,
            dpi=72  # Low DPI for optimal visual tokens
        )

        if not images:
            raise RuntimeError(f"Could not render page {local_page_num} of {pdf_name}")

        buf = io.BytesIO()
        images[0].save(buf, format="PNG")
        buf.seek(0)
        return base64.b64encode(buf.read()).decode("utf-8")

Writing active_rag_engine.py


In [25]:
# @title visual_searching_agent
%%writefile visual_searching_agent.py
# ---
# jupyter:
#   jupytext:
#     formats: ipynb,py:percent
#     text_representation:
#       extension: .py
#       format_name: percent
#       format_version: '1.3'
#       jupytext_version: 1.19.1
#   kernelspec:
#     display_name: Python 3
#     language: python
#     name: python3
# ---

# %% [markdown]
# # Gemma 4 31B — Visual Chess Self-Play (Advanced searching Edition)

# %%
import os
import sys
import json
import time
import io
import re
import datetime
import base64
import hashlib
import threading
import concurrent.futures
from active_rag_engine import NCERTBookManager

# %% tags=["parameters"]
CEREBRAS_API_KEY = "PLACEHOLDER"
MODEL_ID = "gemma-4-31b"
EXTRACT_DIR = "/content"
SEARCH_QUERY = "solve exercises queation no 9 from chapter 10 of class 12 and q 12 from chapter 4 and do the explanation in more detail"

# %% Setup workspace directories & Directory-Scoped Memory
WORKSPACE_DIR = "."
query_slug = re.sub(r'[^a-zA-Z0-9]', '_', SEARCH_QUERY.lower())[:30]
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
SESSION_DIR = os.path.join(WORKSPACE_DIR, "runs", f"run_{timestamp}_{query_slug}")
os.makedirs(SESSION_DIR, exist_ok=True)

folder_path_slug = re.sub(r'[^a-zA-Z0-9]', '_', os.path.abspath(EXTRACT_DIR).lower())[-40:]
MEMORY_FILE_PATH = os.path.join(WORKSPACE_DIR, "runs", f"memory_scope_{folder_path_slug}.txt")
MEMORY_LIMIT = 2200
ENTRY_DELIMITER = "\n§\n"

# Thread safety lock for parallel orchestrations
graph_lock = threading.Lock()

# %% Token Usage Monitor
class TokenUsageMonitor:
    def __init__(self, tpm_limit=100000, context_limit=131072):
        self.tpm_limit = tpm_limit
        self.context_limit = context_limit
        self.total_prompt_tokens = 0
        self.total_completion_tokens = 0
        self.total_calls = 0

    def log_call(self, usage):
        if not usage: return
        with graph_lock:
            self.total_calls += 1
            self.total_prompt_tokens += usage.prompt_tokens
            self.total_completion_tokens += usage.completion_tokens
            print(f"\n📊 Total Session Tokens: {self.total_prompt_tokens + self.total_completion_tokens:,} over {self.total_calls} calls")

MONITOR = TokenUsageMonitor()

# %% Persistent Memory System
def load_searching_memory() -> str:
    with graph_lock:
        if not os.path.exists(MEMORY_FILE_PATH): return "None."
        with open(MEMORY_FILE_PATH, "r", encoding="utf-8") as f: return f.read().strip() or "None."

def _compute_fingerprint(text: str) -> str:
    return hashlib.sha256(" ".join(text.lower().split()).encode("utf-8")).hexdigest()[:16]

def searching_memory(action: str, content: str = None, old_text: str = None) -> str:
    action = action.strip().lower()
    with graph_lock:
        entries = []
        if os.path.exists(MEMORY_FILE_PATH):
            with open(MEMORY_FILE_PATH, "r", encoding="utf-8") as f:
                raw = f.read().strip()
                if raw: entries = [e.strip() for e in raw.split("§") if e.strip()]
        if action == "add":
            new_content = content.strip()
            new_fp = _compute_fingerprint(new_content)
            for existing in entries:
                if _compute_fingerprint(existing.split("] ", 1)[-1] if "] " in existing else existing) == new_fp:
                    return "Notice: Entry already exists."
            if len(ENTRY_DELIMITER.join(entries)) + len(new_content) > MEMORY_LIMIT: return "Error: Full."
            entries.append(f"[{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {new_content}")
        elif action in ("replace", "remove"):
            matches = [i for i, entry in enumerate(entries) if old_text.strip().lower() in entry.lower()]
            if len(matches) != 1: return "Error: Match failure."
            if action == "replace": entries[matches[0]] = f"[{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {content.strip()}"
            else: entries.pop(matches[0])
        with open(MEMORY_FILE_PATH, "w", encoding="utf-8") as f:
            f.write(ENTRY_DELIMITER.join(entries) + ENTRY_DELIMITER if entries else "")
        return "Success."

# %% Client Initialization
from cerebras.cloud.sdk import Cerebras
api_key_to_use = os.environ.get("CEREBRAS_API_KEY") if CEREBRAS_API_KEY == "PLACEHOLDER" else CEREBRAS_API_KEY
client = Cerebras(api_key=api_key_to_use)
book = NCERTBookManager(extract_dir=EXTRACT_DIR)
trajectory_graph = {"nodes": [], "links": []}

def add_node(node_id, label, type, page_info, text="", b64_img_data=None, prompt_tokens=0, completion_tokens=0):
    with graph_lock:
        if not any(n["id"] == node_id for n in trajectory_graph["nodes"]):
            trajectory_graph["nodes"].append({"id": node_id, "label": label, "type": type, "page_info": page_info, "text": text, "b64_img_data": b64_img_data, "prompt_tokens": prompt_tokens, "completion_tokens": completion_tokens, "timestamp": time.time()})

def add_link(source, target, relationship, latency):
    with graph_lock:
        trajectory_graph["links"].append({"source": source, "target": target, "relationship": relationship, "latency_ms": int(latency * 1000)})

# %% Consolidated Metaprompt Instructions
BASE_CORE_INSTRUCTIONS = (
    "\n\nCRITICAL OPERATIONS & CONSTRAINTS:\n"
    "1. Local page translation formula: local_page = printed_page - chapter_start_page + 1. Always evaluate this first.\n"
    "2. Break down queries into 5 Systematic Pillars: Definitions, Taxonomy, Examples, Contrasts/Boundary limits, and Q&A Application items.\n"
    "3. Cross-examine attached images concurrently. Execute memory_action='add' to capture key terms.\n"
    "4. Dynamically append discoveries to updated_queries_list and modify selected_template for optimal routing."
)

PROMPT_TEMPLATES = {
    "SCAN_INDEX": "You are the Unsupervised Directory Scanning Function. Identify chapter/indices roles.",
    "PARSER_TOC": "You are the Table of Contents Parser Function. Map page boundaries." + BASE_CORE_INSTRUCTIONS,
    "EXPLORER_CONCEPT": "You are the Core Pillar Concept Exploration and Search Dissection Function." + BASE_CORE_INSTRUCTIONS,
    "LOOKBACK_TRACER": "You are the Foundational Lookback Tracer Function. Identify origin concepts." + BASE_CORE_INSTRUCTIONS
}

# %% Structured Schemas
directory_scan_schema = {
    "type": "object", "properties": {"thinking_process": {"type": "string"}, "extracted_file_mappings": {"type": "array", "items": {"type": "object", "properties": {"pdf_filename": {"type": "string"}, "identified_chapter_or_role": {"type": "string"}}, "required": ["pdf_filename", "identified_chapter_or_role"], "additionalProperties": False}}}, "required": ["thinking_process", "extracted_file_mappings"], "additionalProperties": False
}

sliding_window_schema = {
    "type": "object", "properties": {"thought": {"type": "string"}, "semantic_connection_reasoning": {"type": "string"}, "updated_queries_list": {"type": "array", "items": {"type": "string"}}, "selected_template": {"type": "string", "enum": ["SCAN_INDEX", "PARSER_TOC", "EXPLORER_CONCEPT", "LOOKBACK_TRACER"]}, "memory_action": {"type": "string", "enum": ["add", "replace", "remove", "none"]}, "memory_content": {"type": "string"}, "memory_old_text": {"type": "string"}, "semantic_link_source": {"type": "string"}, "semantic_link_target": {"type": "string"}, "semantic_link_reason": {"type": "string"}, "spawn_explorer_agent": {"type": "object", "properties": {"pdf_name": {"type": "string"}, "page_num": {"type": "integer"}, "concept_to_extract": {"type": "string"}}, "required": ["pdf_name", "page_num", "concept_to_extract"], "additionalProperties": False}, "pages_to_load": {"type": "array", "items": {"type": "object", "properties": {"pdf_name": {"type": "string"}, "page_num": {"type": "integer"}}, "required": ["pdf_name", "page_num"], "additionalProperties": False}}, "task_complete": {"type": "boolean"}}, "required": ["thought", "semantic_connection_reasoning", "updated_queries_list", "selected_template", "memory_action", "memory_content", "memory_old_text", "semantic_link_source", "semantic_link_target", "semantic_link_reason", "pages_to_load", "task_complete"], "additionalProperties": False
}

# %% Swarm Cloning Agent & Network Core Callers
def spawn_child_searching_agent(pdf_name: str, page_num: int, concept: str, parent_node_id: str):
    t0 = time.time()
    b64_img = book.get_pdf_page_image_b64(pdf_name, page_num)
    response = client.chat.completions.create(model=MODEL_ID, messages=[{"role": "system", "content": f"Extract definition of '{concept}'."}, {"role": "user", "content": [{"type": "text", "text": "Extract."}, {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64_img}"}}]}], temperature=0.1)
    MONITOR.log_call(response.usage)
    txt = response.choices[0].message.content.strip()
    searching_memory("add", content=f"'{concept}': {txt[:100]}")

    child_id = f"child_{concept.lower().replace(' ', '_')}_{parent_node_id}"
    add_node(child_id, f"Child: '{concept}'", "child_agent", f"{pdf_name}:p{page_num}", txt, b64_img, response.usage.prompt_tokens, response.usage.completion_tokens)
    add_link(parent_node_id, child_id, "delegated_to_child_agent", time.time() - t0)

def call_forager_api_with_retries(system_prompt: str, prompt_text: str, images_payload: list, file_name: str, page_num: int):
    for attempt in range(3):
        try:
            user_payload = [{"type": "text", "text": prompt_text}] + [{"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img}"}} for img in images_payload]
            response = client.chat.completions.create(model=MODEL_ID, messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_payload}], response_format={"type": "json_schema", "json_schema": {"name": "searching_decision", "strict": True, "schema": sliding_window_schema}}, temperature=0.1, reasoning_effort="high")
            MONITOR.log_call(response.usage)
            return response, json.loads(response.choices[0].message.content)
        except Exception: time.sleep(2)
    return None, {"thought": "Failed.", "semantic_connection_reasoning": "Failed.", "updated_queries_list": [SEARCH_QUERY], "selected_template": "EXPLORER_CONCEPT", "memory_action": "none", "memory_content": "", "memory_old_text": "", "semantic_link_source": "", "semantic_link_target": "", "semantic_link_reason": "", "pages_to_load": [{"pdf_name": file_name, "page_num": page_num + 1}], "task_complete": False}

def discover_directory_agent(batch: list, images_payload: list) -> tuple:
    payload = [{"type": "text", "text": "Scan covers."}] + [{"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img}"}} for img in images_payload]
    res = client.chat.completions.create(model=MODEL_ID, messages=[{"role": "system", "content": "Map files."}, {"role": "user", "content": payload}], response_format={"type": "json_schema", "json_schema": {"name": "directory_mappings", "strict": True, "schema": directory_scan_schema}}, temperature=0.1, reasoning_effort="high")
    MONITOR.log_call(res.usage)
    return json.loads(res.choices[0].message.content), res.usage

# %% Isolated Query Pipeline Worker
def forage_query_branch(sub_query: str, branch_id: int):
    index_file = "lebo1ps.pdf"
    for f in book.available_files:
        if any(k in f.lower() for k in ["ps", "pre", "index", "toc", "contents"]):
            index_file = f
            break

    active_requests = [{"pdf_name": index_file, "page_num": 11}, {"pdf_name": index_file, "page_num": 12}]
    step, max_steps, active_template, current_queries = 0, 6, "PARSER_TOC", [sub_query]

    while step < max_steps and active_requests:
        step += 1
        req_labels = ", ".join([f"{r['pdf_name']} p{r['page_num']}" for r in active_requests])
        imgs = [book.get_pdf_page_image_b64(r["pdf_name"], r["page_num"]) for r in active_requests]

        prompt_text = f"Branch: {branch_id}. Pages: [{req_labels}]. Active Queries: {current_queries}\nMemory: {load_searching_memory()}"
        res, decision = call_forager_api_with_retries(PROMPT_TEMPLATES[active_template], prompt_text, imgs, active_requests[0]["pdf_name"], active_requests[0]["page_num"])

        current_queries = decision.get("updated_queries_list", current_queries)
        active_template = decision.get("selected_template", "EXPLORER_CONCEPT")
        p_tok = res.usage.prompt_tokens if res else 0
        c_tok = res.usage.completion_tokens if res else 0

        node_id = f"branch_{branch_id}_step_{step}_forage"
        add_node(node_id, f"B{branch_id} Step {step}: [{req_labels}]", "chapter", req_labels, decision["thought"], imgs[0], p_tok, c_tok)

        if step > 1:
            prev_node_id = f"branch_{branch_id}_step_{step-1}_forage"
            add_link(prev_node_id, node_id, "explore_step", 0.5)

        if decision.get("memory_action", "none") != "none":
            searching_memory(decision["memory_action"], content=decision["memory_content"], old_text=decision["memory_old_text"])

        if decision.get("semantic_link_source") and decision.get("semantic_link_target"):
            src, tgt, rsn = decision["semantic_link_source"], decision["semantic_link_target"], decision["semantic_link_reason"]
            s_id, t_id = f"b{branch_id}_c_{src.lower().replace(' ','_')}", f"b{branch_id}_c_{tgt.lower().replace(' ','_')}"
            add_node(s_id, f"Concept: '{src}'", "concept", req_labels, rsn)
            add_node(t_id, f"Concept: '{tgt}'", "concept", req_labels, rsn)
            add_link(s_id, t_id, "semantic_connection", 0.1)

        if decision.get("spawn_explorer_agent"):
            s_req = decision["spawn_explorer_agent"]
            spawn_child_searching_agent(s_req["pdf_name"], s_req["page_num"], s_req["concept_to_extract"], node_id)

        if decision.get("task_complete", False) or not decision.get("pages_to_load"): break
        active_requests = decision["pages_to_load"][:3]

# %% Main Concurrent Execution Simulator Loop
def run_searching_simulation():
    # Phase 1: Sequential Global Directory Discovery (Run once)
    batches = [book.available_files[i:i + 3] for i in range(0, len(book.available_files), 3)]
    for batch in batches[:2]:
        imgs = [book.get_pdf_page_image_b64(file, 1) for file in batch]
        scan, usage = discover_directory_agent(batch, imgs)
        for idx, m in enumerate(scan["extracted_file_mappings"]):
            searching_memory("add", content=f"File {m['pdf_filename']} mapped to {m['identified_chapter_or_role']}")
            add_node(f"discovery_{m['pdf_filename'].replace('.pdf','')}", f"Mapped: {m['pdf_filename']}", "index", f"{m['pdf_filename']}:p1", m['identified_chapter_or_role'], imgs[idx], int(usage.prompt_tokens/3), int(usage.completion_tokens/3))

    # Phase 2: Parallelized Swarm Branching
    sub_queries = re.split(r'\s+and\s+|\s+AND\s+', SEARCH_QUERY)
    print(f"🚀 Deploying {len(sub_queries)} parallel orchestrations concurrently: {sub_queries}")

    with concurrent.futures.ThreadPoolExecutor(max_workers=len(sub_queries)) as executor:
        futures = [executor.submit(forage_query_branch, q, idx + 1) for idx, q in enumerate(sub_queries)]
        concurrent.futures.wait(futures)

    with open(os.path.join(SESSION_DIR, "trajectory_graph.json"), "w") as f: json.dump(trajectory_graph, f, indent=2)
    with open("trajectory_graph.json", "w") as f: json.dump(trajectory_graph, f, indent=2)

run_searching_simulation()

Overwriting visual_searching_agent.py


In [26]:
!jupytext --to notebook visual_searching_agent.py

[jupytext] Reading visual_searching_agent.py in format py
[jupytext] Writing visual_searching_agent.ipynb (destination file replaced [use --update to preserve cell outputs and ids])
[jupytext] Updating the timestamp of visual_searching_agent.py


In [38]:
from google.colab import userdata
cerebras_key = userdata.get('Cerebras_KEY')

!papermill visual_searching_agent.ipynb output_searching_run.ipynb \
  -p CEREBRAS_API_KEY "{cerebras_key}" \
  -p MODEL_ID "gemma-4-31b" \
  -p EXTRACT_DIR "/content/chapters" \
  -p SEARCH_QUERY "solve exercises queation no 9 from c hapter 10 of class 12 "

Input Notebook:  visual_searching_agent.ipynb
Output Notebook: output_searching_run.ipynb
Executing:   0% 0/13 [00:00<?, ?cell/s]Executing notebook with kernel: python3
Executing: 100% 13/13 [00:38<00:00,  2.96s/cell]


In [39]:
# @title  View Visual searching Execution Logs & Token Usage Reports

# %%
import json

def display_searching_logs(notebook_path="output_searching_run.ipynb"):
    try:
        with open(notebook_path, "r", encoding="utf-8") as f:
            nb = json.load(f)
    except FileNotFoundError:
        print(f"❌ Output notebook '{notebook_path}' not found. Please execute Cell 3 first.")
        return

    print("=" * 80)
    print(f"📖 PRINTING ACTIVE searching CELL EXECUTION LOGS FROM: {notebook_path}")
    print("=" * 80 + "\n")

    for idx, cell in enumerate(nb.get("cells", [])):
        if cell.get("cell_type") == "code":
            outputs = cell.get("outputs", [])
            source_lines = cell.get("source", [])
            first_line = "".join(source_lines).split("\n")[0][:60] if source_lines else "Empty Cell"

            logs = []
            has_logs = False
            for out in outputs:
                if out.get("output_type") == "stream" and out.get("name") == "stdout":
                    logs.append("".join(out.get("text", [])))
                    has_logs = True
                elif out.get("output_type") == "error":
                    logs.append(f"❌ EXECUTION ERROR:\n" + "".join(out.get("traceback", [])))
                    has_logs = True

            if has_logs:
                print(f"🔹 [Cell {idx:02d}] {first_line}...")
                print("".join(logs).strip())
                print("-" * 80 + "\n")

display_searching_logs()

📖 PRINTING ACTIVE searching CELL EXECUTION LOGS FROM: output_searching_run.ipynb

🔹 [Cell 07] from cerebras.cloud.sdk import Cerebras...
📂 Successfully mapped 20 active textbook PDFs in /content/chapters:
  • kebo101.pdf
  • kebo102.pdf
  • kebo103.pdf
  • kebo104.pdf
  • kebo105.pdf
  • kebo106.pdf
  • kebo107.pdf
  • kebo108.pdf
  • kebo109.pdf
  • kebo110.pdf
  • kebo111.pdf
  • kebo112.pdf
  • kebo113.pdf
  • kebo114.pdf
  • kebo115.pdf
  • kebo116.pdf
  • kebo117.pdf
  • kebo118.pdf
  • kebo119.pdf
  • kebo1ps.pdf
--------------------------------------------------------------------------------

🔹 [Cell 12] def run_searching_simulation():...
📊 Total Session Tokens: 1,514 over 1 calls

📊 Total Session Tokens: 3,250 over 2 calls
🚀 Deploying 1 parallel orchestrations concurrently: ['solve exercises queation no 9 from c hapter 10 of class 12 ']

📊 Total Session Tokens: 6,628 over 3 calls
⚠️ Warning: Requested page 13 exceeds max pages (11) for kebo110.pdf. Defaulting to last page.

📊 T

In [40]:

# @title Render Interactive D3.js Search Trajectory Graph with Live Token HUD
# @title Render Interactive D3.js Search Trajectory Graph with Live Token HUD

import json
from IPython.display import HTML, display

def render_trajectory_graph(json_path="trajectory_graph.json"):
    try:
        with open(json_path, "r") as f:
            graph_data = json.load(f)
    except FileNotFoundError:
        print("❌ Graph data not found. Please run the visual searching simulation first.")
        return

    # Dynamic D3.js template
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <script src="https://d3js.org/d3.v6.min.js"></script>
        <style>
            #graph-container {{
                background: linear-gradient(135deg, #ffffff 0%, #f0f9ff 50%, #e0f2fe 100%);
                border-radius: 12px;
                padding: 15px;
                border: 1px solid #bae6fd;
                position: relative;
                overflow: hidden;
            }}
            .control-bar {{
                margin-left: 10px;
                margin-bottom: 12px;
                display: flex;
                align-items: center;
                gap: 12px;
            }}
            #play-btn {{
                background-color: #0284c7;
                color: white;
                border: none;
                padding: 8px 16px;
                font-family: sans-serif;
                font-size: 13px;
                font-weight: bold;
                border-radius: 6px;
                cursor: pointer;
                box-shadow: 0 2px 4px rgba(2, 132, 199, 0.2);
                transition: background-color 0.2s, transform 0.1s;
                display: inline-flex;
                align-items: center;
                gap: 6px;
                z-index: 15;
                position: relative;
            }}
            #play-btn:hover {{
                background-color: #0369a1;
            }}
            #play-btn:active {{
                transform: scale(0.97);
            }}
            .token-hud {{
                position: absolute;
                top: 15px;
                right: 15px;
                background: rgba(255, 255, 255, 0.85);
                backdrop-filter: blur(6px);
                -webkit-backdrop-filter: blur(6px);
                border: 1px solid #bae6fd;
                border-radius: 8px;
                padding: 10px 14px;
                font-family: sans-serif;
                font-size: 12px;
                color: #1e293b;
                box-shadow: 0 4px 12px rgba(2, 132, 199, 0.08);
                z-index: 10;
                pointer-events: none;
            }}
            .hud-title {{
                font-weight: bold;
                margin-bottom: 6px;
                color: #0369a1;
                display: flex;
                align-items: center;
                gap: 4px;
            }}
            .hud-metrics {{
                display: flex;
                flex-direction: column;
                gap: 3px;
            }}
            .node-group {{
                cursor: grab;
            }}
            .node-group:active {{
                cursor: grabbing;
            }}
            .link {{
                stroke: #0284c7;
                stroke-opacity: 0.6;
                stroke-width: 2.5px;
                fill: none;
            }}
            .node-text {{
                font-family: sans-serif;
                font-size: 11px;
                fill: #1e293b;
                font-weight: bold;
                pointer-events: none;
            }}
            .tooltip {{
                position: absolute;
                background-color: #ffffff;
                border: 1px solid #cbd5e1;
                color: #1e293b;
                padding: 10px;
                border-radius: 8px;
                font-family: sans-serif;
                font-size: 12px;
                pointer-events: none;
                max-width: 320px;
                box-shadow: 0 4px 12px rgba(0,0,0,0.1);
                z-index: 100;
                opacity: 0;
            }}
        </style>
    </head>
    <body>
        <div id="graph-container">
            <h3 style="color: #0369a1; font-family: sans-serif; margin-left: 10px; margin-bottom: 2px;">🧠 Agent Visual Search & Trajectory Path</h3>
            <p style="color: #64748b; font-family: sans-serif; font-size: 12px; margin-left: 10px; margin-top: 0; margin-bottom: 12px;">
                🖱️ <strong>Controls:</strong> Drag background to pan | Scroll to zoom | Double-click background to reset view.
            </p>

            <div class="control-bar">
                <button id="play-btn">▶️ Play Discovery Animation</button>
            </div>

            <!-- Floating Real-Time Token Dashboard HUD -->
            <div class="token-hud">
                <div class="hud-title">📊 Session Token Usage</div>
                <div class="hud-metrics">
                    <div>📥 Input Tokens: <span id="hud-input-tokens" style="font-weight: 600;">0</span></div>
                    <div>📤 Completion: <span id="hud-completion-tokens" style="font-weight: 600;">0</span></div>
                    <div style="border-top: 1px solid #bae6fd; margin-top: 4px; padding-top: 4px; font-weight: bold; color: #0284c7;">
                        ✨ Total Volume: <span id="hud-total-tokens">0</span>
                    </div>
                </div>
            </div>

            <svg id="canvas" width="900" height="500"></svg>
        </div>
        <div id="tooltip" class="tooltip"></div>

        <script>
            const data = {json.dumps(graph_data)};
            const svg = d3.select("#canvas");
            const width = +svg.attr("width");
            const height = +svg.attr("height");
            const tooltip = d3.select("#tooltip");

            // 1. Create a global container group to hold all zoomable/pannable visual elements
            const globalGroup = svg.append("g").attr("class", "graph-viewport");

            // 2. Define the Orbital Zoom & Pan behavior
            const zoomBehavior = d3.zoom()
                .scaleExtent([0.1, 5])
                .on("zoom", (event) => {{
                    globalGroup.attr("transform", event.transform);
                }});

            svg.call(zoomBehavior);

            svg.on("dblclick.zoom", null);
            svg.on("dblclick", () => {{
                svg.transition().duration(750).call(zoomBehavior.transform, d3.zoomIdentity);
            }});

            const simulation = d3.forceSimulation(data.nodes)
                .force("link", d3.forceLink(data.links).id(d => d.id).distance(160))
                .force("charge", d3.forceManyBody().strength(-400))
                .force("center", d3.forceCenter(width / 2, height / 2));

            const defs = svg.append("defs");

            defs.selectAll("clipPath")
                .data(data.nodes)
                .enter().append("clipPath")
                .attr("id", d => "clip-" + d.id)
                .append("circle")
                .attr("r", 20);

            const link = globalGroup.append("g")
                .selectAll("path")
                .data(data.links)
                .enter().append("path")
                .attr("class", "link");

            const node = globalGroup.append("g")
                .selectAll(".node-group")
                .data(data.nodes)
                .enter().append("g")
                .attr("class", "node-group")
                .call(d3.drag()
                    .on("start", dragstarted)
                    .on("drag", dragged)
                    .on("end", dragended));

            node.append("image")
                .attr("xlink:href", d => {{
                    if (d.b64_img_data) {{
                        return "data:image/png;base64," + d.b64_img_data;
                    }}
                    if (d.type === "index") return "https://img.icons8.com/color/96/book.png";
                    if (d.type === "target_chapter") return "https://img.icons8.com/color/96/opened-folder-key.png";
                    if (d.type === "child_agent") return "https://img.icons8.com/color/48/bot.png";
                    return "https://img.icons8.com/color/96/light.png";
                }})
                .attr("x", -20)
                .attr("y", -20)
                .attr("width", 40)
                .attr("height", 40)
                .attr("clip-path", d => "url(#clip-" + d.id + ")");

            const text = globalGroup.append("g")
                .selectAll("text")
                .data(data.nodes)
                .enter().append("text")
                .attr("class", "node-text")
                .attr("dx", 26)
                .attr("dy", 4)
                .text(d => d.label);

            node.on("mouseover", (event, d) => {{
                // Corrected token computation helpers
                const pTokens = d.prompt_tokens || (d.text ? Math.round(d.text.length * 0.25) + 1150 : 1150);
                const cTokens = d.completion_tokens || (d.text ? Math.round(d.text.length * 0.35) + 40 : 400);

                tooltip.transition().duration(200).style("opacity", .98);
                tooltip.html(`
                    <strong style="color:#0284c7;">Node:</strong> ${{d.label}}<br/>
                    <strong style="color:#b45309;">Location:</strong> ${{d.page_info || 'Concept'}}<br/>
                    <strong style="color:#0369a1;">Tokens Used:</strong> ${{(pTokens + cTokens).toLocaleString()}} <span style="font-size:10px; color:#64748b;">(📥${{pTokens}} / 📤${{cTokens}})</span><br/>
                    <hr style="border: 0; border-top: 1px solid #cbd5e1; margin: 6px 0;"/>
                    <strong>Extract / Rationale:</strong><br/>
                    <span style="font-size:11px; color:#475569;">${{d.text ? d.text.substring(0, 250) : 'N/A'}}...</span>
                `)
                .style("left", (event.pageX + 15) + "px")
                .style("top", (event.pageY - 28) + "px");
            }})
            .on("mousemove", (event) => {{
                tooltip.style("left", (event.pageX + 15) + "px")
                       .style("top", (event.pageY - 28) + "px");
            }})
            .on("mouseout", () => {{
                tooltip.transition().duration(500).style("opacity", 0);
            }});

            simulation.on("tick", () => {{
                link.attr("d", d => `M${{d.source.x}},${{d.source.y}} L${{d.target.x}},${{d.target.y}}`);
                node.attr("transform", d => `translate(${{d.x}},${{d.y}})`);
                text.attr("x", d => d.x).attr("y", d => d.y);
            }});

            // --- ANIMATION ENGINE LOGIC WITH PROGRESSIVE TOKEN HUD TRACKING ---
            let animationTimeouts = [];

            d3.select("#play-btn").on("click", () => {{
                // Clear any running animation timeouts
                animationTimeouts.forEach(t => clearTimeout(t));
                animationTimeouts = [];

                // 1. Center view frame smoothly
                svg.transition()
                    .duration(750)
                    .call(zoomBehavior.transform, d3.zoomIdentity);

                // 2. Clear out element visibilities instantly
                node.style("opacity", 0);
                link.style("opacity", 0);
                text.style("opacity", 0);

                // 3. Reset Token Tally HUD Values
                let currentInputTotal = 0;
                let currentCompletionTotal = 0;
                document.getElementById("hud-input-tokens").innerText = "0";
                document.getElementById("hud-completion-tokens").innerText = "0";
                document.getElementById("hud-total-tokens").innerText = "0";

                // 4. Staggered reveal based on processing sequence (600ms steps)
                data.nodes.forEach((d, i) => {{
                    const timeoutId = setTimeout(() => {{
                        // Fade in Node & Label
                        node.filter(n => n.id === d.id).transition().duration(400).style("opacity", 1);
                        text.filter(n => n.id === d.id).transition().duration(400).style("opacity", 1);

                        // Fade in associated pathway links
                        link.filter(l => l.target.id === d.id || l.source.id === d.id)
                            .transition().duration(400).style("opacity", 0.6);

                        // Dynamically pull or compute token payloads per node action
                        const nodePromptTokens = d.prompt_tokens || (d.text ? Math.round(d.text.length * 0.25) + 1150 : 1150);
                        const nodeCompletionTokens = d.completion_tokens || (d.text ? Math.round(d.text.length * 0.35) + 40 : 400);

                        currentInputTotal += nodePromptTokens;
                        currentCompletionTotal += nodeCompletionTokens;

                        // Increment Dashboard Elements Counter display
                        document.getElementById("hud-input-tokens").innerText = currentInputTotal.toLocaleString();
                        document.getElementById("hud-completion-tokens").innerText = currentCompletionTotal.toLocaleString();
                        document.getElementById("hud-total-tokens").innerText = (currentInputTotal + currentCompletionTotal).toLocaleString();

                    }}, i * 600);

                    animationTimeouts.push(timeoutId);
                }});
            }});

            function dragstarted(event, d) {{
                if (!event.active) simulation.alphaTarget(0.3).restart();
                d.fx = d.x;
                d.fy = d.y;
            }}

            function dragged(event, d) {{
                d.fx = event.x;
                d.fy = event.y;
            }}

            function dragended(event, d) {{
                if (!event.active) simulation.alphaTarget(0);
                d.fx = null;
                d.fy = null;
            }}
        </script>
    </body>
    </html>
    """
    display(HTML(html_content))

render_trajectory_graph()